# APTOS 2019 — D1 + H1 + H5 ensemble submission

Logit-averaging ensemble of three exp300-architecture checkpoints:
- **D1** (`exp300`) — ImageNet init
- **H1** (`exp701`) — EyePACS OrdSupCon backbone init
- **H5** (`exp705`) — APTOS OrdSupCon (A1-v3) backbone init

All three share the same architecture (ResNet50 + GeM + Dropout(0.3)+Linear) and the same
Ben-Graham preprocessing — they only differ in how the backbone was initialized before
fine-tuning. Diverse inits → ensemble usually adds +0.005 to +0.02 QWK over best single.

**Before running:** upload each checkpoint as a Kaggle dataset and edit `CHECKPOINT_PATHS`
in the Configuration cell to match the `/kaggle/input/<slug>` paths Kaggle assigns.


In [ ]:
# Cell 1 — imports & device
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}, Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2 — configuration
NUM_CLASSES = 5
IMAGE_SIZE  = 512
BATCH_SIZE  = 32
NUM_WORKERS = 2

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# === EDIT THESE FOR YOUR KAGGLE DATASETS ===
# Upload each checkpoint folder as a separate Kaggle dataset (or one combined dataset).
# The keys ('d1', 'h1', 'h5') are just labels for logging.
CHECKPOINT_PATHS = {
    'd1': '/kaggle/input/exp300-d1-dropout-cosine/exp300_d1_dropout_cosine_best.pth',
    'h1': '/kaggle/input/exp701-h1-ordsupcon-d1recipe/exp701_h1_ordsupcon_d1recipe_best.pth',
    'h5': '/kaggle/input/exp705-h5-a1-d1recipe/exp705_h5_a1_d1recipe_best.pth',
}

TEST_CSV        = '/kaggle/input/aptos2019-blindness-detection/test.csv'
TEST_IMAGES_DIR = '/kaggle/input/aptos2019-blindness-detection/test_images'
OUTPUT_PATH     = '/kaggle/working/submission.csv'

# Architecture hyperparameters (must match training — identical across D1/H1/H5)
HEAD_DROPOUT = 0.3
GEM_P        = 3.0


In [ ]:
# Cell 3 — model: ResNet50 + GeM + Dropout(0.3)+Linear head (mirrors src/models.py)
class GeM(nn.Module):
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1)),
        ).pow(1.0 / self.p)


def build_exp300_model():
    # weights=None — checkpoint will overwrite, and Kaggle's torchvision has known issues
    # loading ImageNet weights (see notebooks/kaggle_runner.ipynb).
    model = tvm.resnet50(weights=None)
    model.avgpool = GeM(p=GEM_P)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=HEAD_DROPOUT),
        nn.Linear(in_features, NUM_CLASSES),
    )
    return model


In [ ]:
# Cell 4 — Ben-Graham preprocessing (verbatim from src/dataset.py: bbox crop)
def ben_graham_preprocess(image: np.ndarray, size: int = IMAGE_SIZE) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        image = image[y:y + h, x:x + w]
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    gauss = cv2.GaussianBlur(image, (0, 0), sigmaX=size / 30)
    image = cv2.addWeighted(image, 4, gauss, -4, 128)
    return image


In [ ]:
# Cell 5 — test dataset (mirrors DRDataset image-load + normalize sequence)
class APTOSTestDataset(Dataset):
    def __init__(self, csv_path, img_dir, size=IMAGE_SIZE):
        self.df = pd.read_csv(csv_path)         # 'id_code' column
        self.img_dir = img_dir
        self.size = size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        code = self.df.iloc[idx]['id_code']
        image = cv2.imread(os.path.join(self.img_dir, f'{code}.png'))   # BGR
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = ben_graham_preprocess(image, self.size)
        image_t = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        for c in range(3):
            image_t[c] = (image_t[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
        return image_t, code


In [ ]:
# Cell 6 — load all three checkpoints
def load_model(ckpt_path: str) -> nn.Module:
    m = build_exp300_model().to(DEVICE)
    sd = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    if isinstance(sd, dict) and 'model_state_dict' in sd:
        sd = sd['model_state_dict']
    result = m.load_state_dict(sd, strict=True)
    m.eval()
    return m

models = {}
for name, path in CHECKPOINT_PATHS.items():
    models[name] = load_model(path)
    print(f'  [{name}] loaded {os.path.basename(path)}')

total_params = sum(sum(p.numel() for p in m.parameters()) for m in models.values())
print(f'Ensemble: {len(models)} models, total params: {total_params/1e6:.1f}M')


In [ ]:
# Cell 7 — DataLoader
test_ds = APTOSTestDataset(TEST_CSV, TEST_IMAGES_DIR)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f'Test images: {len(test_ds)}')


In [ ]:
# Cell 8 — ensemble inference (logit averaging across D1 + H1 + H5)
all_ids, all_preds = [], []
with torch.no_grad():
    for images, codes in tqdm(test_loader, desc='Ensemble inference'):
        images = images.to(DEVICE, non_blocking=True)
        # Sum raw logits across the 3 models, then argmax (logit averaging).
        # Equivalent to averaging since argmax is scale-invariant within a row.
        ensemble_logits = torch.zeros(images.size(0), NUM_CLASSES, device=DEVICE)
        for m in models.values():
            ensemble_logits += m(images)
        preds = ensemble_logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_ids.extend(codes)

dist = pd.Series(all_preds).value_counts().sort_index().to_dict()
print(f'Total predictions: {len(all_preds)}')
print(f'Ensemble grade distribution: {dist}')


In [ ]:
# Cell 9 — write submission
submission = pd.DataFrame({'id_code': all_ids, 'diagnosis': all_preds})
submission.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote {OUTPUT_PATH}  shape={submission.shape}')
submission.head()
